# Assignment 5.1 GRU training smoke test

A custom randomly initialized PyTorch GRU was trained for one epoch on the
existing primary outer training partition. This notebook displays the saved
actual evidence and verifies the checkpoint. It can optionally launch a new
run without overwriting earlier evidence.

Select a Python 3.11+ kernel with PyTorch (the existing rocm-jupyter environment
has CPU PyTorch), restart once and Run All. Launch from project root or notebooks/.
Restore the existing data/radio and result manifests first. No pretrained
model, held-out performance or vendor-restriction result is claimed.

[Full configuration and executed evidence](../docs/ASSIGNMENT_5_1_EVIDENCE.md).


In [1]:
from pathlib import Path
import sys, json, hashlib
root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / 'src/reliability/assignment51_gru.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Launch from project root or notebooks/.')
if str(root / 'src') not in sys.path:
    sys.path.insert(0, str(root / 'src'))
import torch
from reliability.assignment51_gru import GRUBinaryClassifier, FEATURES, run_smoke, parameter_hash
print('Interpreter:', sys.executable)
print('PyTorch:', torch.__version__)
print('Training device for this demonstration: CPU (explicit)')
print('Feature order:', ', '.join(FEATURES))
print('Source SHA-256:', hashlib.sha256((root/'src/reliability/assignment51_gru.py').read_bytes()).hexdigest())


Interpreter: C:\Users\saiga\miniforge3\envs\rocm-jupyter\python.exe
PyTorch: 2.9.1+cpu
Training device for this demonstration: CPU (explicit)
Feature order: dlMcs, dlBler, ulMcs, ulBler, phr, pcmax, rsrq, sinr, rsrp, rssi, cqi, pucchSnr, puschSnr
Source SHA-256: a4b98cd59f7826ea68191d145b82eac823b94583acab2dd2a08a849e736c2f19


## Architecture

Input is batch x at-most-ten reports x 13 numeric features. A one-layer,
unidirectional GRU has 32 hidden units and dropout 0. Its final true hidden
state feeds a linear layer producing one raw attack logit. Packed lengths
exclude padded tail positions. Each window starts with zero state. The model
is a custom nn.Module using PyTorch layers, not a manual reimplementation of
GRU kernels. No pretrained weights are loaded.

BCEWithLogitsLoss supplies the sigmoid/log-loss combination internally; no
extra sigmoid is applied before loss. Input mean/population-SD come only from
26,455 primary training reports. IDs, timestamp values, labels, constants/PMI
and unresolved byte counters are excluded from the fixed assignment input list.


In [2]:
model = GRUBinaryClassifier()
print(model)
print('Trainable parameters:', sum(p.numel() for p in model.parameters()))


GRUBinaryClassifier(
  (gru): GRU(13, 32, batch_first=True)
  (dropout): Dropout(p=0.0, inplace=False)
  (output): Linear(in_features=32, out_features=1, bias=True)
)
Trainable parameters: 4545


## Actual execution evidence

The default below reads the completed CLI run; it does not train again.
To demonstrate training interactively, set RUN_NEW_TRAINING=True. A new dated
unique result directory will be created. The default example/checkpoint remain
untouched. CPU training is sufficient; a GPU/ROCm runtime is not required.


In [3]:
RUN_NEW_TRAINING = False
run_dir = root / 'results/assignment_5_1_gru_smoke_2026-11-24'
if RUN_NEW_TRAINING:
    from datetime import datetime, timezone
    from uuid import uuid4
    run_dir = root / 'results' / ('assignment_5_1_gru_smoke_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid4().hex[:8])
    run_smoke(root, run_dir, epochs=1)
record = json.loads((run_dir / 'training_evidence.json').read_text(encoding='utf-8'))
checkpoint = Path(record['checkpoint'])
if not checkpoint.is_absolute():
    checkpoint = root / checkpoint
assert hashlib.sha256(checkpoint.read_bytes()).hexdigest() == record['checkpoint_sha256']
restored = GRUBinaryClassifier()
restored.load_state_dict(torch.load(checkpoint, map_location='cpu', weights_only=True)['model_state_dict'])
assert parameter_hash(restored) == record['trained_parameter_sha256']
assert record['initial_parameter_sha256'] != record['trained_parameter_sha256']
print('Run classification:', record['experiment'])
print('Configuration:', json.dumps(record['config'], indent=2))
print('Epoch history:', json.dumps(record['history'], indent=2))
print('Reports/windows/groups:', record['training_reports'], record['training_windows'], len(record['audit']['training_group_ids']))
print('Window class counts (0=Benign, 1=attack):', record['window_class_counts'])
print('Partial windows retained:', record['audit']['partial_training_windows'])
print('Checkpoint:', checkpoint)
print('Checkpoint checksum/reload and changed weights verified.')


Run classification: SMOKE TEST ONLY; no held-out or vendor-restriction result
Configuration: {
  "batch_size": 64,
  "bidirectional": false,
  "cpu_threads": 1,
  "deterministic_algorithms": true,
  "device": "cpu",
  "dropout": 0.0,
  "epochs": 1,
  "features": [
    "dlMcs",
    "dlBler",
    "ulMcs",
    "ulBler",
    "phr",
    "pcmax",
    "rsrq",
    "sinr",
    "rsrp",
    "rssi",
    "cqi",
    "pucchSnr",
    "puschSnr"
  ],
  "hidden_size": 32,
  "input_size": 13,
  "layers": 1,
  "learning_rate": 0.001,
  "loss": "BCEWithLogitsLoss, unweighted mean",
  "max_reports": 10,
  "num_workers": 0,
  "optimizer": "Adam",
  "output": "one raw attack logit; no sigmoid before loss",
  "seed": 42
}
Epoch history: [
  {
    "epoch": 1,
    "optimizer_steps": 42,
    "seconds": 0.30494419997558,
    "training_loss": 0.571900883934479,
    "windows_seen": 2669
  }
]
Reports/windows/groups: 26455 2669 24
Window class counts (0=Benign, 1=attack): {'0': 615, '1': 2054}
Partial windows retaine

## Interpretation and remaining work

The loss measures optimization on training windows, not held-out accuracy or
security performance. Source-file groups are disjoint but true independent
capture IDs remain unverified. The run is SMOKE TEST ONLY. There is no vendor
capability/profile experiment, threshold, calibration, abstention execution or
claim that these features are the minimum/required telemetry set. Nitendra
retains ownership of final models and calibration. Continue capability/input
agreement and the reviewed evaluation protocol before confirmatory experiments.
